# UniConv-EvalKit Tutorial

**UniConv-EvalKit** is a unified framework for **evaluating multi-turn interactions in Large Language Models (LLMs)**, covering dimensions such as **memory, understanding, safety, mathematics, and code**.

This tutorial will walk you through the core concepts and usage of the framework:

1. **Architecture Overview** — Understand core components and data flow
2. **Data Structures** — Learn about `Dialog`, `Turn`, `MetricConfig` and other core schemas
3. **Data Loading & Exploration** — Load and inspect dialog data using `BenchmarkDataset`
4. **Python API** — Run the full evaluation pipeline via `EvalPipeline`
5. **Phase-by-Phase Execution** — Run Data → Generation → Evaluation → Aggregation on demand
6. **CLI Usage** — Quickly launch evaluation tasks from the command line
7. **Result Aggregation & Analysis** — Understand multi-level aggregation strategies


---
## 1. Architecture Overview

UniConv-EvalKit follows a modular design with fully decoupled core components:

```
src/
├── dataset/          # Dataset loading and preprocessing
├── metric/           # Metric implementations (Exact Match, LLM Judge, etc.)
├── model/            # Model interface wrappers (OpenAI, etc.)
├── config.py         # EvalPipelineConfig configuration
├── phases.py         # Four-phase logic (Data / Generation / Evaluation / Aggregation)
├── eval_pipeline.py  # EvalPipeline main controller
└── eval_cli.py       # CLI entry point
```

**The evaluation workflow consists of four phases:**

```
┌─────────────┐    ┌──────────────────┐    ┌──────────────────┐    ┌──────────────────┐
│  DataPhase   │ →  │ GenerationPhase  │ →  │ EvaluationPhase  │ →  │ AggregationPhase │
│  Preprocess  │    │  Model Response  │    │  Metric Scoring  │    │  Aggregation     │
└─────────────┘    └──────────────────┘    └──────────────────┘    └──────────────────┘
```


In [1]:
import sys, os
import warnings
warnings.filterwarnings("ignore")

# Ensure the project root is in the Python path
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/kiki/Documents/mypapers/EvolKit/code


---
## 2. Core Data Structures

Understanding the data structures is fundamental to using this framework. Let's take a look at the core schema definitions.


### Data Structure Diagram

```
Dialog
├── dialog_id: int                    # Dialog ID
├── dialog_labels: Dict               # Dialog-level labels (task_type, task_subtype, etc.)
├── dialog_eval_config
│   └── use_reference_history: bool   # Whether to use reference answers to build history
└── dialog_turns: List[Turn]
    ├── Turn (role="user")
    │   ├── turn_id, content
    │   └── eval_config.do_eval = False
    └── Turn (role="assistant")
        ├── turn_id, content (model generated), reference (ground truth)
        ├── reference_document        # (optional) Ground truth document or history turn_id
        └── eval_config
            ├── do_eval: True / False
            ├── metrics: [MetricConfig(class_name, args)] # Metrics and their args to use if evaluated
            └── dynamic_config_source:   # Args for dynamically adjusting eval config at runtime; alternative to static `metrics`
```


In [ ]:
from src.dataset.schema import Dialog, Turn, TurnEvalConfig, MetricConfig, DialogEvalConfig

# ========== 2.1 MetricConfig — Metric Configuration ==========
# Each turn to be evaluated can have one or more metrics configured
metric_cfg = MetricConfig(
    class_name="numeric_match",  # Metric name, corresponds to a key in METRIC_REGISTRY
    args={}                      # Additional arguments passed to the metric
)
print("MetricConfig example:")
print(metric_cfg.model_dump_json(indent=2))


MetricConfig example:
{
  "class_name": "numeric_match",
  "args": {}
}


In [ ]:
# ========== 2.2 TurnEvalConfig — Turn-Level Evaluation Config ==========
turn_eval_cfg = TurnEvalConfig(
    do_eval=True,            # Whether to evaluate this turn
    metrics=[metric_cfg],    # Which metrics to use
)
print("TurnEvalConfig example:")
print(turn_eval_cfg.model_dump_json(indent=2))


TurnEvalConfig example:
{
  "do_eval": true,
  "metrics": [
    {
      "class_name": "numeric_match",
      "args": {}
    }
  ],
  "dynamic_config_source": {}
}


In [ ]:
# ========== 2.3 Turn — Dialog Turn ==========
user_turn = Turn(
    turn_id=0,
    role="user",
    content="What is 1+1?",
)

assistant_turn = Turn(
    turn_id=1,
    role="assistant",
    content=None,                        # To be generated by the model
    reference="1+1=2, the answer is 2",  # Reference answer
    eval_config=turn_eval_cfg,           # Requires evaluation
)

print("User Turn:")
print(user_turn.model_dump_json(indent=2))
print("\nAssistant Turn (to be generated / evaluated):")
print(assistant_turn.model_dump_json(indent=2))


User Turn:
{
  "turn_id": 0,
  "role": "user",
  "content": "What is 1+1?",
  "reference": null,
  "reference_document": null,
  "eval_config": {
    "do_eval": false,
    "metrics": [],
    "dynamic_config_source": {}
  },
  "turn_labels": {}
}

Assistant Turn (to be generated):
{
  "turn_id": 1,
  "role": "assistant",
  "content": null,
  "reference": "1+1=2, the answer is 2",
  "reference_document": null,
  "eval_config": {
    "do_eval": true,
    "metrics": [
      {
        "class_name": "numeric_match",
        "args": {}
      }
    ],
    "dynamic_config_source": {}
  },
  "turn_labels": {}
}


In [ ]:
# ========== 2.4 Dialog — Complete Dialog ==========
dialog = Dialog(
    dialog_id=0,
    dialog_labels={"task_type": "math"},     # Dialog-level labels (used for aggregation analysis)
    dialog_eval_config=DialogEvalConfig(
        use_reference_history=True             # Use reference answers as history during generation
    ),
    dialog_turns=[user_turn, assistant_turn],
)

print("Dialog example:")
print(dialog.model_dump_json(indent=2))


Dialog example:
{
  "dialog_id": 0,
  "dialog_raw_info": {},
  "dialog_labels": {
    "task_type": "math"
  },
  "dialog_eval_config": {
    "use_reference_history": true
  },
  "dialog_turns": [
    {
      "turn_id": 0,
      "role": "user",
      "content": "What is 1+1?",
      "reference": null,
      "reference_document": null,
      "eval_config": {
        "do_eval": false,
        "metrics": [],
        "dynamic_config_source": {}
      },
      "turn_labels": {}
    },
    {
      "turn_id": 1,
      "role": "assistant",
      "content": null,
      "reference": "1+1=2, the answer is 2",
      "reference_document": null,
      "eval_config": {
        "do_eval": true,
        "metrics": [
          {
            "class_name": "numeric_match",
            "args": {}
          }
        ],
        "dynamic_config_source": {}
      },
      "turn_labels": {}
    }
  ]
}


---
## 3. Data Loading & Exploration

This framework supports a variety of evaluation datasets. Let's see which datasets are available and how to load and inspect them.


In [2]:
# ========== 3.1 List All Registered Datasets ==========
from src.dataset import DATASET_REGISTRY, get_dataset_class

print("Registered datasets:")
print("=" * 50)
for name, cls in DATASET_REGISTRY.items():
    print(f"  • {name:20s} → {cls.__name__}")
print(f"\nTotal: {len(DATASET_REGISTRY)} datasets")


Registered datasets:
  • mt_eval              → MTEvalDataset
  • multi_challenge      → MultiChallengeDataset
  • mt_bench_101         → MTBench101Dataset
  • multi_if             → MultiIFDataset
  • locomo               → LoCoMoDataset
  • longmemeval          → LongMemEvalDataset
  • personamem           → PersonaMemDataset
  • mathchat             → MathChatDataset
  • memorycode           → MemoryCodeDataset
  • safedialbench        → SafeDialBenchDataset

Total: 10 datasets


In [ ]:
# ========== 3.2 List All Registered Metrics ==========
from src.metric import METRIC_REGISTRY

print("Registered metrics:")
print("=" * 50)
for name, cls in METRIC_REGISTRY.items():
    print(f"  • {name:25s} → {cls.__name__}")
print(f"\nTotal: {len(METRIC_REGISTRY)} metrics")


Registered metrics:
  • precision                 → PrecisionMetric
  • recall                    → RecallMetric
  • exact_match               → ExactMatchMetric
  • f1_score                  → F1Metric
  • instruction_following     → InstructionFollowingMetric
  • llm_judge                 → LLMJudge
  • numeric_match             → NumericMatchMetric
  • code_match                → CodeMatchMetric

Total: 8 metrics


---
## 4. Python API: One-Click Evaluation

Using `EvalPipeline` and `EvalPipelineConfig`, you can flexibly invoke the full evaluation pipeline within scripts or notebooks.


In [ ]:
from src.eval_pipeline import EvalPipeline, EvalPipelineConfig

# ========== 4.1 Configuration ==========
# Note: Running this cell requires a valid API Key and Base URL
# Below is a configuration example (will not actually execute)

cfg = EvalPipelineConfig(
    # --- Dataset ---
    dataset="mt_eval",                     # Dataset name
    raw_data_dir="./raw_data/MT-Eval",     # Path to raw data
    processed_data_dir="./data",            # Path for preprocessed data storage
    output_dir="./output",                  # Output directory
    
    # --- Model Under Evaluation ---
    model_type="openai",                    # Model API type; customizable in src/model/
    model_name="deepseek-v3.2",            # Model to evaluate
    temperature=0.7,                        # Generation temperature
    max_tokens=1024,                        # Maximum tokens
    
    # --- Judge Model (optional) ---
    judge_model_type="openai",              # Judge model API type
    judge_model_name="gpt-4.1",            # For LLM-as-a-Judge
    
    # --- API Configuration ---
    api_key="your-api-key",                 # Replace with your API Key
    base_url="your-base-url",               # API Base URL
    
    # --- Task Control ---
    do_generation=True,                     # Run generation phase
    do_evaluation=True,                     # Run evaluation phase
    parallel=4,                             # Number of parallel threads
    
    # --- Aggregation Strategy ---
    agg_turn_stat="mean",                   # Turn-level multi-metric aggregation: mean/min/max
    agg_dialog_stat="min",                  # Dialog-level multi-turn aggregation: mean/min/max
    agg_dataset_level="dialog",             # Dataset-level aggregation granularity: dialog/turn
)

print("✅ EvalPipelineConfig created successfully")
print(f"   Dataset: {cfg.dataset}")
print(f"   Model: {cfg.model_name}")
print(f"   Judge: {cfg.judge_model_name}")
print(f"   Generation output dir: {cfg.gen_output_dir}")
print(f"   Evaluation output dir: {cfg.eval_output_dir}")
print(f"   Summary file path: {cfg.summary_output_path}")


✅ EvalPipelineConfig created successfully
   Dataset: mt_eval
   Model: deepseek-v3.2
   Judge: gpt-4.1
   Generation output dir: output/mt_eval/deepseek-v3.2/generated
   Evaluation output dir: output/mt_eval/deepseek-v3.2/eval_details
   Summary file path: output/mt_eval/deepseek-v3.2/summary.json


In [ ]:
# ========== 4.2 One-Click Run (requires valid API Key) ==========

# ⚠️ Uncomment the lines below to actually run the evaluation
# pipeline = EvalPipeline(cfg)
# pipeline.run()

print("💡 Uncomment the lines above and configure a valid API Key to run the full evaluation.")
print("   pipeline.run() will execute the following phases in sequence:")
print("   1. prepare_data()    — Data preprocessing")
print("   2. run_generation()  — Model generation")
print("   3. run_evaluation()  — Metric evaluation")
print("   4. run_aggregation() — Result aggregation")


Evaluating: 100%|██████████| 10/10 [00:31<00:00,  3.11s/it]


Global Results:
{
  "count": 10,
  "mean": 0.7,
  "min": 0.0,
  "max": 1.0
}



---
## 5. Phase-by-Phase Execution

`EvalPipeline` supports invoking each phase independently, allowing for custom processing between stages.


In [ ]:
# ========== 5.1 Phase 1: Data Preparation ==========
# This step does not require an API Key and can be run directly

cfg_data_only = EvalPipelineConfig(
    dataset="mt_eval",
    raw_data_dir="./raw_data/MT-Eval",
    processed_data_dir="./data",
)

pipeline = EvalPipeline(cfg_data_only)

# If the raw_data directory exists, run data preprocessing
from pathlib import Path

raw_data_path = Path(cfg_data_only.raw_data_dir)
if raw_data_path.exists():
    dialogs = pipeline.prepare_data()
    print(f"✅ Successfully loaded {len(dialogs)} dialogs")
    
    # View summary of the first dialog
    if dialogs:
        d = dialogs[0]
        print(f"\nFirst dialog:")
        print(f"  ID: {d.dialog_id}")
        print(f"  Labels: {d.dialog_labels}")
        print(f"  Number of turns: {len(d.dialog_turns)}")
else:
    # If raw data doesn't exist, load from preprocessed data
    data_dir = Path("./data/mt_eval")
    if data_dir.exists():
        dialogs = list(pipeline.dataset.load_eval_dialogs(
            data_root=str(data_dir), recursive=True
        ))
        print(f"✅ Loaded {len(dialogs)} dialogs from preprocessed data")
    else:
        print("⚠️ Neither raw data nor preprocessed data exists. Please prepare the data first.")
        dialogs = []


✅ Successfully loaded 10 dialogs

First dialog:
  ID: 0
  Labels: {'task_type': 'expansion', 'task_subtype': 'multi'}
  Number of turns: 14


In [ ]:
# ========== 5.2 Phase 2: Model Generation (requires API Key) ==========

# Re-create pipeline with model & API config for generation
cfg_gen = EvalPipelineConfig(
    dataset="mt_eval",
    raw_data_dir="./raw_data/MT-Eval",
    processed_data_dir="./data",
    output_dir="./output",
    model_type="openai",
    model_name="deepseek-v3.2",           # e.g. "gpt-3.5-turbo", "deepseek-chat"
    api_key="your-api-key",                 # Replace with your API Key
    base_url="your-base-url",   # Replace with your API Base URL
    parallel=4,
)
pipeline_gen = EvalPipeline(cfg_gen)

# ⚠️ Uncomment the line below to actually run generation
# generated_dialogs = pipeline_gen.run_generation(dialogs)

print("💡 Phase 2: Model Generation")
print("   This phase calls the model to generate responses for each assistant turn where do_eval=True.")
print("   Supports checkpoint resume — already generated dialogs will be automatically skipped.")
print(f"   Results are saved to: output/{cfg_gen.dataset}/{cfg_gen.model_name}/generated/")


💡 Phase 2: Model Generation
   This phase calls the model to generate responses for each assistant turn where do_eval=True.
   Supports checkpoint resume — already generated dialogs will be automatically skipped.
   Results are saved to: output/mt_eval/deepseek-v3.2/generated/


In [ ]:
# ========== 5.3 Phase 3: Metric Evaluation (requires API Key if using LLM Judge) ==========

# Only need output path + judge model config (if using LLM Judge metrics)
cfg_eval = EvalPipelineConfig(
    dataset="mt_eval",
    model_name="deepseek-v3.2",               # Must match the model used in generation
    output_dir="./output",
    parallel=4,
    # --- Only required if the dataset uses LLM Judge ---
    judge_model_type="openai",                   # Judge model API type
    judge_model_name="gpt-4.1",                 # Judge model name
    api_key="your-api-key",                      # API Key for the judge model
    base_url="your-base-url", # ,        # API Base URL for the judge model
)
pipeline_eval = EvalPipeline(cfg_eval)

# ⚠️ Uncomment the line below to actually run evaluation
# results = pipeline_eval.run_evaluation(generated_dialogs)

print("💡 Phase 3: Metric Evaluation")
print("   This phase computes metric scores for each generated turn.")
print(f"   Results are saved to: output/{cfg_eval.dataset}/{cfg_eval.model_name}/eval_details/")


💡 Phase 3: Metric Evaluation
   This phase computes metric scores for each generated turn.
   Results are saved to: output/mt_eval/deepseek-v3.2/eval_details/


In [ ]:
# ========== 5.4 Phase 4: Result Aggregation ==========

# Configure output path and aggregation strategy
cfg_agg = EvalPipelineConfig(
    dataset="mt_eval",
    model_name="deepseek-v3.2",               # Must match the model used in generation
    output_dir="./output",
    agg_turn_stat="mean",                        # Turn-level: mean/min/max
    agg_dialog_stat="min",                       # Dialog-level: mean/min/max
    agg_dataset_level="dialog",                  # Dataset-level granularity: dialog/turn
)
pipeline_agg = EvalPipeline(cfg_agg)

# ⚠️ Uncomment the line below to actually run aggregation
# summary = pipeline_agg.run_aggregation(results)

print("💡 Phase 4: Result Aggregation")
print("   Aggregation strategy (fine to coarse):")
print("   ┌──────────────────────────────────────────────────────────────────┐")
print("   │  Turn Level   ──(agg_turn_stat)──→   Combined score per Turn   │")
print("   │  Dialog Level ──(agg_dialog_stat)──→  Combined score per Dialog │")
print("   │  Dataset Level ──(agg_dataset_level)──→ Final dataset score     │")
print("   └──────────────────────────────────────────────────────────────────┘")
print("\n   Available aggregation methods: mean / min / max")
print(f"   Summary saved to: output/{cfg_agg.dataset}/{cfg_agg.model_name}/summary.json")



Global Results:
{
  "count": 10,
  "mean": 0.71,
  "min": 0.0,
  "max": 1.0
}

💡 Phase 4: Result Aggregation
   Aggregation strategy (fine to coarse):
   ┌──────────────────────────────────────────────────────────────────┐
   │  Turn Level   ──(agg_turn_stat)──→   Combined score per Turn   │
   │  Dialog Level ──(agg_dialog_stat)──→  Combined score per Dialog │
   │  Dataset Level ──(agg_dataset_level)──→ Final dataset score     │
   └──────────────────────────────────────────────────────────────────┘

   Available aggregation methods: mean / min / max
   Summary saved to: output/mt_eval/deepseek-v3.2/summary.json


---
## 6. CLI Usage

In addition to the Python API, you can quickly launch evaluation tasks from the command line.


In [3]:
# ========== 6.1 View CLI Help ==========
!PYTHONPATH=. python3 -W ignore src/eval_cli.py --help


usage: eval_cli.py [-h] [--dataset DATASET] [--raw_data_dir RAW_DATA_DIR]
                   [--processed_data_dir PROCESSED_DATA_DIR]
                   [--output_dir OUTPUT_DIR] [--model_type MODEL_TYPE]
                   [--model_name MODEL_NAME] [--temperature TEMPERATURE]
                   [--max_tokens MAX_TOKENS]
                   [--judge_model_type JUDGE_MODEL_TYPE]
                   [--judge_model_name JUDGE_MODEL_NAME] [--parallel PARALLEL]
                   [--api_key API_KEY] [--base_url BASE_URL] [--do_generation]
                   [--do_evaluation] [--agg_by_metric]
                   [--agg_turn_stat {mean,min,max}]
                   [--agg_dialog_stat {mean,min,max}]
                   [--agg_dataset_level {dialog,turn}]

Evaluation Pipeline

optional arguments:
  -h, --help            show this help message and exit
  --dataset DATASET     Benchmark dataset name (e.g., mt_eval)
  --raw_data_dir RAW_DATA_DIR
                        Path to raw dataset files
  --

### Common CLI Command Examples

**Full Workflow (Generation + Evaluation):**
```bash
PYTHONPATH=. python src/eval_cli.py \
    --dataset mt_eval \
    --raw_data_dir ./raw_data/MT-Eval \
    --model_name deepseek-v3.2 \
    --judge_model_name gpt-4.1 \
    --base_url "your-base-url" \
    --api_key "your-api-key" \
    --do_generation \
    --do_evaluation \
    --parallel 4
```

**Generation Only:**
```bash
PYTHONPATH=. python src/eval_cli.py \
    --dataset mathchat \
    --raw_data_dir ./raw_data/MathChat \
    --model_name deepseek-v3.2 \
    --base_url "your-base-url" \
    --api_key "your-api-key" \
    --do_generation
```

**Evaluation Only (requires existing generation results):**
```bash
PYTHONPATH=. python src/eval_cli.py \
    --dataset mathchat \
    --model_name deepseek-v3.2 \
    --judge_model_name gpt-4.1 \
    --base_url "your-base-url" \
    --api_key "your-api-key" \
    --do_evaluation
```

**Using a Local vLLM Model:**
```bash
# First, start the vLLM service
python3 -m vllm.entrypoints.openai.api_server \
    --model ./Models/Qwen3-8B \
    --served-model-name "Qwen3-8B" \
    --port 8000

# Then run the evaluation
PYTHONPATH=. python src/eval_cli.py \
    --dataset locomo \
    --raw_data_dir ./raw_data/LoCoMo \
    --model_name Qwen3-8B \
    --base_url http://localhost:8000/v1/ \
    --do_generation
```


---
## 7. Result Aggregation & Analysis

Let's dive into the aggregation mechanism and how to view and analyze evaluation results.


In [5]:
# ========== 7.1 Mock Evaluation Results & Aggregation Demo ==========
from src.metric.aggregator import aggregate_results

# Construct pseudo evaluation results
pseudo_results = [
    # Dialog 0, Turn 1 — two metrics
    {"dialog_id": 0, "turn_id": 1, "metric_name": "numeric_match",
     "score": 1.0, "details": {"score": 1.0}, 
     "dialog_labels": {"task_type": "follow_up"}, "turn_labels": {}},
    {"dialog_id": 0, "turn_id": 1, "metric_name": "llm_judge",
     "score": 0.8, "details": {"score": 0.8}, 
     "dialog_labels": {"task_type": "follow_up"}, "turn_labels": {}},
    # Dialog 0, Turn 3
    {"dialog_id": 0, "turn_id": 3, "metric_name": "numeric_match",
     "score": 0.0, "details": {"score": 0.0}, 
     "dialog_labels": {"task_type": "follow_up"}, "turn_labels": {}},
    # Dialog 1, Turn 1
    {"dialog_id": 1, "turn_id": 1, "metric_name": "numeric_match",
     "score": 1.0, "details": {"score": 1.0}, 
     "dialog_labels": {"task_type": "correction"}, "turn_labels": {}},
    # Dialog 1, Turn 3
    {"dialog_id": 1, "turn_id": 3, "metric_name": "numeric_match",
     "score": 0.5, "details": {"score": 0.5}, 
     "dialog_labels": {"task_type": "correction"}, "turn_labels": {}},
]

print(f"Number of pseudo evaluation records: {len(mock_results)}")
print(f"Number of Dialogs: {len(set(r['dialog_id'] for r in mock_results))}")
print(f"Number of Turns: {len(set((r['dialog_id'], r['turn_id']) for r in mock_results))}")


Number of pseudo evaluation records: 5
Number of Dialogs: 2
Number of Turns: 4


In [8]:
# ========== 7.2 Using Different Aggregation Strategies ==========
import json 

# Strategy 1: mean-min-dialog (default)
result_1 = aggregate_results(
    mock_results, 
    turn_stat="mean",       # Average multiple metrics within a Turn
    dialog_stat="min",      # Take the minimum across Turns within a Dialog (strictest)
    dataset_level="dialog"  # Average across Dialogs at the dataset level
)

print("Strategy 1: turn=mean, dialog=min, level=dialog")
print(f"  Global stats: {json.dumps(result_1['global'], indent=4)}")
print(f"  By-label stats: {json.dumps(result_1['by_label'], indent=4)}")

print("\n" + "=" * 50)

# Strategy 2: mean-mean-dialog (lenient)
result_2 = aggregate_results(
    mock_results,
    turn_stat="mean",
    dialog_stat="mean",     # Average across Turns within a Dialog (more lenient)
    dataset_level="dialog"
)
print("\nStrategy 2: turn=mean, dialog=mean, level=dialog")
print(f"  Global stats: {json.dumps(result_2['global'], indent=4)}")

print("\n" + "=" * 50)

# Strategy 3: Per-metric statistics
result_3 = aggregate_results(
    mock_results,
    turn_stat="mean",
    dialog_stat="min",
    dataset_level="dialog",
    by_metric=True          # Compute statistics per metric name
)
print("\nStrategy 3: by_metric=True")
print(f"  Per-metric stats: {json.dumps(result_3['by_metric'], indent=4)}")


Strategy 1: turn=mean, dialog=min, level=dialog
  Global stats: {
    "count": 2,
    "mean": 0.25,
    "min": 0.0,
    "max": 0.5
}
  By-label stats: {
    "dialog": {
        "task_type:('correction', 'count')": 1.0,
        "task_type:('correction', 'mean')": 0.5,
        "task_type:('correction', 'min')": 0.5,
        "task_type:('correction', 'max')": 0.5,
        "task_type:('follow_up', 'count')": 1.0,
        "task_type:('follow_up', 'mean')": 0.0,
        "task_type:('follow_up', 'min')": 0.0,
        "task_type:('follow_up', 'max')": 0.0
    },
    "turn": {}
}


Strategy 2: turn=mean, dialog=mean, level=dialog
  Global stats: {
    "count": 2,
    "mean": 0.6,
    "min": 0.45,
    "max": 0.75
}


Strategy 3: by_metric=True
  Per-metric stats: {
    "('llm_judge', 'count')": 1.0,
    "('llm_judge', 'mean')": 0.8,
    "('llm_judge', 'min')": 0.8,
    "('llm_judge', 'max')": 0.8,
    "('numeric_match', 'count')": 4.0,
    "('numeric_match', 'mean')": 0.625,
    "('numeric_match

---
## Summary

This tutorial covered the core usage of UniConv-EvalKit:

1. **Data Structures** — The hierarchical structure of `Dialog` → `Turn` → `TurnEvalConfig` → `MetricConfig`
2. **Data Loading** — Loading different datasets via `BenchmarkDataset` and the registry mechanism
3. **Python API** — Running evaluations with `EvalPipelineConfig` + `EvalPipeline` (one-click or phase-by-phase)
4. **CLI Usage** — Quickly launching evaluations from the command line
5. **Result Aggregation** — Flexible three-level aggregation strategies (Turn → Dialog → Dataset)

### Quick Start Checklist

- [ ] Place raw data in the `raw_data/` directory
- [ ] Configure your API Key and Base URL
- [ ] Select a dataset and model
- [ ] Run `pipeline.run()` or use a CLI command
- [ ] Check results in the `output/` directory

For more information, refer to the [README.md](./README.md) or submit a [GitHub Issue](https://github.com/JiaQiSJTU/UniConv-EvalKit/issues).
